### Lexicon Builder


In [1]:
import os
import sys
import re
import json
import pickle

import requests
import spacy 
import nltk

from collections import Counter
import pandas as pd
from tqdm import tqdm

sys.path.append('..')

from utils.json import *
from utils.dataset import *
from utils.lexicon import *

# TODO Implement manual labeling of languages
# - Alternatives to FastText
# - Manual Labeling
# - Not crucial but useful

tqdm.pandas()


In [2]:
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

In [3]:
# Define the paths
path_annotations = "../data/annotations"
path_lexicons = "../data/lexicons"
path_negation = "../data/lexicons/negation"
path_uncertainty = "../data/lexicons/uncertainty"
path_model = "../data/lexicon/models"

In [4]:
"""
# -- FastText Language Detection (Not works as expected) --

# !pip install fasttext spacy
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

# URL of the FastText model
url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
 
# Path to fastText model
file_model = os.path.join(path_model, "lid.176.bin")

# Download fastText model if needed
if not os.path.exists(file_model):
    print("Downloading fastText language detection model...")
    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an exception
    with open(file_model, "wb") as f:
        f.write(response.content)
    print("Model downloaded successfully!")
else:
    print("FastText model already exists at:", file_model)
"""

'\n# -- FastText Language Detection (Not works as expected) --\n\n# !pip install fasttext spacy\n# !python -m spacy download es_core_news_sm\n# !python -m spacy download ca_core_news_sm\n\n# URL of the FastText model\nurl = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"\n \n# Path to fastText model\nfile_model = os.path.join(path_model, "lid.176.bin")\n\n# Download fastText model if needed\nif not os.path.exists(file_model):\n    print("Downloading fastText language detection model...")\n    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"\n    response = requests.get(url, stream=True)\n    response.raise_for_status() # Raise an exception\n    with open(file_model, "wb") as f:\n        f.write(response.content)\n    print("Model downloaded successfully!")\nelse:\n    print("FastText model already exists at:", file_model)\n'

In [5]:
os.makedirs(path_negation, exist_ok=True)
os.makedirs(path_uncertainty, exist_ok=True)

In [6]:
# Load the training dataframe
with open(os.path.join(path_annotations, "df_train.pkl"), "rb") as f:
    df_train = pickle.load(f)

display(df_train.head())

print("Label distribution:")
print(df_train["label"].value_counts())

,doc_index,doc_id,result_id,start,end,label,text,line_number
0,0,19026587,ent0,448,451,NEG,no,19026587_0
1,0,19026587,ent1,451,467,NSCO,habitos toxicos.,19026587_0
19,0,19026587,ent19,51,59,NEG,"afebril,",19026587_11
2,0,19026587,ent2,286,298,NSCO,cistoscopia,19026587_2
3,0,19026587,ent3,305,314,NEG,negativa,19026587_2


Label distribution:
label
NEG     4307
NSCO    4103
UNC      458
USCO     451
Name: count, dtype: int64


In [7]:
def clean_text(text):
    if not text:
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"^[\.,;:\s]+|[\.,;:\s]+$", "", text) # Remove punctuation at beginning and end
    
    return text

In [8]:
neg_df = df_train[df_train["label"] == "NEG"].copy() # Filter only NEG labels
unc_df = df_train[df_train["label"] == "UNC"].copy() # Filter only UNC labels
print(f"Found {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")


# Obtain unique negation and uncertainty cues
unique_neg_cues = set(neg_df["text"].apply(clean_text).unique())
unique_unc_cues = set(unc_df["text"].apply(clean_text).unique())
print(f"Unique negation cues: {len(unique_neg_cues)}")
print(f"Unique uncertainty cues: {len(unique_unc_cues)}")

Found 4307 negation cues and 458 uncertainty cues
Unique negation cues: 60
Unique uncertainty cues: 79


In [9]:
# Clean and normalize terms
neg_df["clean_text"] = neg_df["text"].apply(clean_text)
unc_df["clean_text"] = unc_df["text"].apply(clean_text)

neg_df = neg_df[neg_df["clean_text"] != ""] # Remove empty terms
unc_df = unc_df[unc_df["clean_text"] != ""] # Remove empty terms
print(f"After cleaning: {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")

neg_counts = neg_df["clean_text"].value_counts().to_dict() # Count term frequencies
unc_counts = unc_df["clean_text"].value_counts().to_dict() # Count term frequencies


neg_lexicon = pd.DataFrame({ # NEG Lexicon DataFrames with unique terms
    "term": list(neg_counts.keys()),
    "freq": list(neg_counts.values())
})

unc_lexicon = pd.DataFrame({ # UNC Lexicon DataFrames with unique terms
    "term": list(unc_counts.keys()),
    "freq": list(unc_counts.values())
})


print("\nTOP negation cues:")
for term, count in sorted(neg_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{term}: {count}")

print("\nTOP uncertainty cues:")
for term, count in sorted(unc_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{term}: {count}")

After cleaning: 4305 negation cues and 458 uncertainty cues

TOP negation cues:
no: 1909
sin: 1423
negativo: 200
afebril: 188
niega: 136
negativos: 92
ausencia de: 64
negativa: 53
sense: 40
neg: 35
negativas: 24
asintomatica: 11
asintomatico: 10
ex: 9
descarta: 8
inespecifico: 8
falta de: 7
impide: 5
ex-: 4
exfumador: 4

TOP uncertainty cues:
compatible con: 58
probable: 54
sospecha de: 31
se orienta: 23
probablemente: 23
posible: 20
sugestiva de: 18
aparentemente: 17
compatibles con: 17
dudosa: 13
valorar: 12
sugestivo de: 10
podria: 10
parece: 9
sugestivos de: 9
sugestivas de: 7
aparente: 7
posiblemente: 5
vs: 5
dudoso: 5


In [10]:
# -- Process lexicons with language detection --

# Recreate words.py even if it exists
force_rebuild = False # Do not change this value unless you know what you are doing


# Normal operation uses existing words.py if available
neg_lexicon, unc_lexicon = process_lexicons_with_language(neg_lexicon, unc_lexicon)


# print("Negation lexicon preview:")
# display(neg_lexicon.head())

# print("Uncertainty lexicon preview:")
# display(unc_lexicon.head())

Loading existing language classifications from utils
Loaded:
    - 72 negation terms
    - 96 uncertainty terms
Found:
    - 0 new negation terms
    - 0 new uncertainty terms
Applying existing classifications to lexicons...
Terms appearing multiple times in negation lexicon: 13
Terms appearing multiple times in uncertainty lexicon: 17
Existing classifications applied successfully!


In [11]:
def determine_POS(term, language):
    """
    Determine the part of speech (POS) for a term
    
    Parameters:
        term (str): The term to analyze
        language (str): Language of the term ("es" or "ca")
        
    Returns:
        str: Part of speech category
    """
    term = term.strip().lower()
    
    # Handle special cases
    if not term:
        return "NA"  # Empty term
    
    # Check for prefixes and suffixes
    prefixes = ["in", "im", "i", "des", "dis", "a"]
    if term in prefixes or term.endswith("-"):
        return "prefix"
    
    if term.startswith("-"):
        return "suffix"
    
    global nlp_es, nlp_ca
    
    nlp_es, nlp_ca = load_spacy_models()
    
    if nlp_es is None or nlp_ca is None:
        return "unknown"
    
    # Select appropriate model
    nlp = nlp_ca if language == "ca" else nlp_es
    
    # Process with spaCy
    doc = nlp(term)
    
    # Map spaCy's universal POS tags to categories
    pos_map = {
        "ADV": "adverb",
        "VERB": "verb",
        "ADP": "preposition",
        "DET": "determiner",
        "ADJ": "adjective",
        "NOUN": "noun",
        "PRON": "pronoun",
        "CCONJ": "conjunction",
        "SCONJ": "conjunction"
    }
    
    # Handle multi-word expressions
    if len(doc) > 1:
        # Look for the root of the phrase
        roots = [token for token in doc if token.dep_ == "ROOT"]
        if roots:
            pos = roots[0].pos_
            return pos_map.get(pos, "phrase")
        else:
            return "phrase"
    
    # Handle single-word expressions
    if len(doc) == 1:
        pos = doc[0].pos_
        return pos_map.get(pos, "other")
    
    return "NA"  # Default fallback

# Apply POS tagging
print("Determining POS for negation terms...")
neg_lexicon["POS"] = neg_lexicon.apply(
    lambda row: determine_POS(row["term"], row["language"]), axis=1
)

print("Determining POS for uncertainty terms...")
unc_lexicon["POS"] = unc_lexicon.apply(
    lambda row: determine_POS(row["term"], row["language"]), axis=1
)

Determining POS for negation terms...
Loading spaCy models...
Both Spanish and Catalan models loaded successfully
Determining POS for uncertainty terms...


In [12]:
# Sort lexicons by frequency
neg_lexicon = neg_lexicon.sort_values("freq", ascending=False).reset_index(drop=True)
unc_lexicon = unc_lexicon.sort_values("freq", ascending=False).reset_index(drop=True)

print("Negation lexicon preview:")
display(neg_lexicon.head())

print("Uncertainty lexicon preview:")
display(unc_lexicon.head())

Negation lexicon preview:


,term,freq,language,POS
0,no,1909,es,adverb
1,no,1909,ca,adverb
2,sin,1423,es,preposition
3,negativo,200,es,adjective
4,afebril,188,es,noun


Uncertainty lexicon preview:


,term,freq,language,POS
0,compatible con,58,es,adjective
1,probable,54,es,adjective
2,probable,54,ca,adjective
3,sospecha de,31,es,verb
4,se orienta,23,es,verb


In [13]:
# Save complete lexicons to directory
neg_lexicon.to_csv(os.path.join(path_negation, "negation_ALL.csv"), index=False)
unc_lexicon.to_csv(os.path.join(path_uncertainty, "uncertainty_ALL.csv"), index=False)

# Language and save
neg_lexicon_es = neg_lexicon[neg_lexicon["language"] == "es"]
neg_lexicon_ca = neg_lexicon[neg_lexicon["language"] == "ca"]
unc_lexicon_es = unc_lexicon[unc_lexicon["language"] == "es"]
unc_lexicon_ca = unc_lexicon[unc_lexicon["language"] == "ca"]

In [14]:
# Save to negation and uncertainty directories
neg_lexicon_es.to_csv(os.path.join(path_negation, "negation_es.csv"), index=False)
neg_lexicon_ca.to_csv(os.path.join(path_negation, "negation_ca.csv"), index=False)
unc_lexicon_es.to_csv(os.path.join(path_uncertainty, "uncertainty_es.csv"), index=False)
unc_lexicon_ca.to_csv(os.path.join(path_uncertainty, "uncertainty_ca.csv"), index=False)

print(f"Saved {len(neg_lexicon_es)} Spanish and {len(neg_lexicon_ca)} Catalan negation terms")
print(f"Saved {len(unc_lexicon_es)} Spanish and {len(unc_lexicon_ca)} Catalan uncertainty terms")

# Output language and POS distributions
print("\nNegation language distribution:")
print(neg_lexicon["language"].value_counts())

print("\nNegation POS distribution:")
print(neg_lexicon["POS"].value_counts())

print("\nUncertainty language distribution:")
print(unc_lexicon["language"].value_counts())

print("\nUncertainty POS distribution:")
print(unc_lexicon["POS"].value_counts())

print("\nLexicon building complete!")

Saved 55 Spanish and 17 Catalan negation terms
Saved 76 Spanish and 20 Catalan uncertainty terms

Negation language distribution:
language
es    55
ca    17
Name: count, dtype: int64

Negation POS distribution:
POS
noun           20
adjective      18
verb           16
adverb          4
other           4
preposition     3
prefix          2
phrase          2
determiner      2
pronoun         1
Name: count, dtype: int64

Uncertainty language distribution:
language
es    76
ca    20
Name: count, dtype: int64

Uncertainty POS distribution:
POS
adjective      40
verb           34
noun           10
adverb          8
other           2
pronoun         1
preposition     1
Name: count, dtype: int64

Lexicon building complete!
